In [ ]:
# [Setup]: 
#  Had to install minoconda (Windows): https://www.anaconda.com/download/success
# Had to install Microsoft Visual Code, check Desktop development with C++ in install page: https://visualstudio.microsoft.com/visual-cpp-build-tools/
# Had to install Python extension in VSCodium

# Had to run in Anaconda prompt: 
    # conda create -n rcbplates python=3.11 -y
    # conda activate rcbplates
    # conda install -c conda-forge astropy photutils matplotlib numpy scipy pillow pycairo cairo pkg-config -y
    # pip install daschlab

# Then, in VSCodium:
    # Ctrl + Shift + P -> Python: Select Interpreter -> Python 3.11 (rcbplates)


In [ ]:
pip install daschlab astropy photutils matplotlib numpy scipy pillow pycairo cairo pkg-config astroquery

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement cairo (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\dapur\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for cairo


In [ ]:
# NOTE: Running this will take a while, depending on how many plates are being queried.

from IPython.display import HTML, display

# FIX: white overlay behind tqdm text for readability
display(HTML("""
<style>
/* Make ALL output text white */
.output, .output_text, .output_stream, .output_stdout, 
.cell-output, .cell-output-print, .cell-output-stdout {
    color: white !important;
}

/* Make all text in outputs white including warnings */
.output * {
    color: white !important;
}

/* Specifically target warning messages */
.output .ansi-yellow-fg, .output .ansi-yellow-foreground,
.warning, .Warning, .warnings {
    color: #FFD700 !important;  /* Golden yellow for warnings */
    text-shadow: 0px 0px 5px rgba(255, 215, 0, 0.3);
}

/* tqdm notebook text styling */
.progress-bar,
.tqdm,
.tqdm div,
.tqdm span,
.tqdm .desc,
.tqdm .bar,
.tqdm .progress,
.tqdm .unit,
.tqdm .rate,
.tqdm .eta,
.tqdm .percentage,
.tqdm .bar_cont {
    color: white !important;
    text-shadow:
        0px 0px 3px rgba(255,255,255,0.9),
        0px 0px 6px rgba(255,255,255,0.6);
}

/* Make ALL tqdm text elements white */
.tqdm * {
    color: white !important;
}

/* optional subtle white "overlay glow" behind labels */
.tqdm .desc,
.tqdm .unit,
.tqdm .rate,
.tqdm .eta {
    background: rgba(0, 0, 0, 0.3);
    padding: 2px 6px;
    border-radius: 4px;
}

/* keep bar visible and set bar color */
.tqdm .bar {
    background-color: #4CAF50 !important;
}

/* keep progress text white */
.tqdm .progress-text {
    color: white !important;
}

/* dark background for better visibility */
.tqdm {
    background: rgba(0, 0, 0, 0.2);
    padding: 4px;
    border-radius: 4px;
}

/* Style for regular print statements */
.output pre, .output code {
    color: white !important;
}
</style>
"""))

from daschlab import open_session
from pathlib import Path
import pandas as pd
import numpy as np

from astropy.io import fits
from astropy.wcs import WCS
from astropy.stats import sigma_clipped_stats
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.vizier import Vizier
from photutils.detection import DAOStarFinder
from tqdm.notebook import tqdm

session_dir = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB"
)

session_dir.mkdir(parents=True, exist_ok=True)

sess = open_session(str(session_dir))

sess.select_target("R CrB")
sess.select_refcat("apass")  # AAVSO Photometric All-Sky Survey

exposures = sess.exposures()
print(f"Total exposures: {len(exposures)}")

Vizier.ROW_LIMIT = -1

photometry_rows = []

cutout_dir = session_dir / "cutouts"

for i in tqdm(
    range(min(300, len(exposures))), # NOTE: Change this to range(len(exposures)) to download all plates, range(min(number, len(exposures))) to download number plates
    desc="Downloading cutouts",
    unit="plate"
):
    try:
        sess.cutout(i)

    except Exception as e:
        print(f"\nSkipping exposure {i}: cannot download cutout ({e})")
        continue

# IMPORTANT: now process ONLY actual FITS files
cutouts = sorted(cutout_dir.glob("*.fits"))
print(f"Found {len(cutouts)} cutout files")

for fits_path in tqdm(
    cutouts,
    desc="Processing FITS",
    unit="file"
):

    try:
        hdu = fits.open(fits_path)[0]
        image = hdu.data
        wcs = WCS(hdu.header)

        mean, median, std = sigma_clipped_stats(image)

        finder = DAOStarFinder(threshold=5.0 * std, fwhm=3.0)
        sources = finder(image - median)

        if sources is None:
            continue

        coords = SkyCoord.from_pixel(
            sources["x_centroid"],
            sources["y_centroid"],
            wcs
        )

        center = SkyCoord(
            hdu.header["CRVAL1"],
            hdu.header["CRVAL2"],
            unit="deg"
        )

        apass = Vizier.query_region(
            center,
            radius=10 * u.arcmin,
            catalog="II/336/apass9"
        )[0]

        apass_coords = SkyCoord(
            apass["RAJ2000"],
            apass["DEJ2001"],
            unit="deg"
        )

        idx, sep2d, _ = coords.match_to_catalog_sky(apass_coords)
        matched = sep2d < 3 * u.arcsec

        matched_bmags = []

        for j in range(len(sources)):
            if matched[j]:
                matched_bmags.append(apass["Bmag"][idx[j]])

        plate_apass_B_mag = (
            np.nanmedian(matched_bmags)
            if len(matched_bmags) > 0
            else np.nan
        )

        exp_id = fits_path.name

        row = {
            "fits_file": exp_id,
            "plate_apass_B_mag": plate_apass_B_mag,
            "n_matched_stars": len(matched_bmags)
        }

        photometry_rows.append(row)

    except Exception as e:
        print(f"\nSkipping file {fits_path}: {e}")

photometry_df = pd.DataFrame(photometry_rows)

csv_path = session_dir / "R_CrB_B_magnitudes.csv"
photometry_df.to_csv(csv_path, index=False)

print(f"Saved photometry table to: {csv_path}")
print(photometry_df.head())

- Querying API ...
- Retrieved 15217 relevant exposures in 25 seconds and saved as `exposures.ecsv`
Total exposures: 15217


- Querying API ...


- Fetched 1399680 bytes in 6 seconds

Skipping exposure 101: cannot download cutout (Illegal value: masked.)
Found 57 cutout files


Processing FITS:   0%|          | 0/57 [00:00<?, ?file/s]


Skipping file C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB\cutouts\00022_i01319m1s0.fits: 'DEJ2001'


...


Skipping file C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB\cutouts\00172_i18249m1s0.fits: 'DEJ2001'



Skipping file C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB\cutouts\00174_i18280m1s0.fits: 'DEJ2001'

Skipping file C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB\cutouts\00178_i20684m1s0.fits: 'DEJ2001'



Skipping file C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB\cutouts\00181_a03046m1s0.fits: 'DEJ2001'

Skipping file C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB\cutouts\00182_a03056m0s0.fits: 'DEJ2001'

Skipping file C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB\cutouts\00184_i20874m1s0.fits: 'DEJ2001'



Skipping file C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB\cutouts\00187_i20969m1s0.fits: 'DEJ2001'


Saved photometry table to: C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB\R_CrB_B_magnitudes.csv
Empty DataFrame
Columns: []
Index: []


In [6]:
from pathlib import Path
from io import BytesIO
from astropy.io import fits
from astropy.time import Time
from astropy.stats import sigma_clipped_stats
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# photutils is needed for the linearity/reciprocity-failure check (star
# detection + aperture photometry). Guarded import so the rest of the
# notebook still works if it's not installed -- pip install photutils
try:
    from photutils.detection import DAOStarFinder
    from photutils.aperture import CircularAperture, CircularAnnulus, ApertureStats, aperture_photometry
    PHOTUTILS_AVAILABLE = True
except ImportError:
    PHOTUTILS_AVAILABLE = False
    print("photutils not found -- run 'pip install photutils' to enable the Linearity Check toggle.")

# Dark styling for the dropdown + toggles + table + kill any default borders/backgrounds/outlines
display(HTML("""
<style>

.dark-dropdown select {
    background-color: white !important;
    color: black !important;
    border: 1px solid #555 !important;
}

.dark-toggle {
    color: white !important;
    background-color: #333 !important;
    border: 1px solid #777 !important;
    font-weight: bold !important;
    font-size: 13px !important;
    padding: 6px 10px !important;
}

.dark-toggle:hover {
    background-color: #4a4a4a !important;
}

/* Applied on top of dark-toggle when a toggle is switched on, for visibility */
.dark-toggle-active {
    background-color: #2e7d32 !important;
    border: 1px solid #66bb6a !important;
}

.dark-toggle-active:hover {
    background-color: #388e3c !important;
}

.dark-output,
.dark-output .output,
.dark-output .widget-output,
.jp-OutputArea-output,
.jp-OutputArea-child,
.cell-output-ipywidget-background,
.output,
.output_wrapper,
.widgetarea,
.jp-RenderedHTMLCommon,
.jupyter-widgets {
    background-color: #111 !important;
    border: none !important;
    box-shadow: none !important;
    outline: none !important;
}

body, .jp-Notebook, .jp-WindowedPanel-outer, .jp-Cell-outputArea {
    background-color: #111 !important;
}

/* Make sure any stray stderr/warning text stays readable and wraps instead
   of forcing the whole page wider with one long unwrapped line */
.jp-OutputArea-output[data-mime-type="application/vnd.jupyter.stderr"],
.output_stderr,
.output_stderr pre,
pre {
    color: #ddd !important;
    white-space: pre-wrap !important;
    word-wrap: break-word !important;
    overflow-wrap: break-word !important;
}

.dark-table {
    border-collapse: collapse;
    color: #ddd;
    background-color: #111;
    font-family: monospace;
    font-size: 11px;
}

.dark-table th,
.dark-table td {
    border: 1px solid #333;
    padding: 2px 6px;
    text-align: right;
}

.dark-table th {
    background-color: #222;
    color: white;
}

.table-scroll-box {
    max-height: 500px;
    max-width: 100%;
    overflow: auto;
    background-color: #111;
}

/* Header table styling */
.header-table {
    border-collapse: collapse;
    color: #ddd;
    background-color: #111;
    font-family: monospace;
    font-size: 12px;
    width: 100%;
}

.header-table th,
.header-table td {
    border: 1px solid #333;
    padding: 4px 10px;
    text-align: left;
}

.header-table th {
    background-color: #222;
    color: white;
}

.header-scroll-box {
    max-height: 300px;
    max-width: 100%;
    overflow: auto;
    background-color: #111;
    margin-bottom: 20px;
}

</style>
"""))

cutout_dir = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB\cutouts"
)

csv_path = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB\R_CrB_B_magnitudes.csv"
)

photometry_df = pd.read_csv(csv_path)

cutouts = sorted(cutout_dir.glob("*.fits"))[:50] #NOTE: Add [:number] to process number plates. Remove this to process all that exist in cutout_dir
print(f"Found {len(cutouts)} cutouts")

# Diagnostic (how many CSV rows actually have a usable B magnitude)
total_rows = len(photometry_df)
valid_mask = photometry_df["plate_apass_B_mag"].notna() & (photometry_df["n_matched_stars"] > 0)
valid_rows = valid_mask.sum()

print(f"CSV rows total: {total_rows}")
print(f"CSV rows with a non-nan B mag AND matched_stars > 0: {valid_rows}")
print(f"CSV rows with nan B mag or 0 matched stars: {total_rows - valid_rows}")

if valid_rows == 0:
    print("WARNING: every row in the CSV currently has nan/0 matched stars. "
          "This points to an issue upstream in however plate_apass_B_mag / "
          "n_matched_stars were originally computed (e.g. APASS catalog match "
          "radius too tight, wrong magnitude column pulled from APASS, or the "
          "matching step silently failing for all plates) rather than anything "
          "in this notebook cell.")

# Minimum matched stars (from the CSV) for the dropdown to flag a plate as
# having enough info for a meaningful linearity check. This is just a cheap
# proxy from existing data -- it does NOT run star detection on every plate,
# that still only happens lazily when you open the Linearity Check toggle.
MIN_STARS_FOR_LINEARITY = 3

# Pull an observation date out of the header (falls back to filename if none found)
def get_plate_date(path):
    try:
        hdr = fits.getheader(path)
    except Exception:
        return path.stem

    for key in ("DATE-OBS", "DATE_OBS", "DATEORIG"):
        if key in hdr and hdr[key]:
            return str(hdr[key])

    for key in ("MJD",):
        if key in hdr and hdr[key]:
            try:
                return Time(float(hdr[key]), format="mjd").iso.split(" ")[0]
            except Exception:
                pass

    for key in ("JD",):
        if key in hdr and hdr[key]:
            try:
                return Time(float(hdr[key]), format="jd").iso.split(" ")[0]
            except Exception:
                pass

    return path.stem

# Build a lookup of filename -> (bmag, n_matched), normalized once
photometry_df["fits_file"] = photometry_df["fits_file"].astype(str).str.strip()
bmag_lookup = photometry_df.set_index("fits_file")[["plate_apass_B_mag", "n_matched_stars"]].to_dict("index")

# Build dropdown options as "date - filename - Bmag status - Linearity status"
# NOTE: the Bmag checkmark means "has a usable (non-nan, matched) B magnitude",
# not just "appears in the CSV at all", those are different things.
# The Linearity flag is just based on how many matched stars exist in the CSV
# for that plate -- it's a cheap proxy, not the actual slope-fit result.

dropdown_options = []

for f in cutouts:

    filename = str(f.name).strip()

    entry = bmag_lookup.get(filename)

    if entry is None:
        bmag_label = "✖ Not in CSV"
        linearity_label = "✖ No data"
    elif pd.isna(entry["plate_apass_B_mag"]) or entry["n_matched_stars"] == 0:
        bmag_label = "△ In CSV, no match"
        linearity_label = "✖ No stars"
    else:
        bmag_label = "✔ Bmag"
        n_matched = entry["n_matched_stars"]
        if n_matched >= MIN_STARS_FOR_LINEARITY:
            linearity_label = "✔ Linearity OK"
        else:
            linearity_label = "△ Few stars"

    dropdown_options.append(
        (f"{get_plate_date(f)} - {filename}  |  {bmag_label}  |  {linearity_label}", f)
    )

dropdown = widgets.Dropdown(
    options=dropdown_options,
    description="",
    layout=widgets.Layout(width="1200px")
)

dropdown.add_class("dark-dropdown")

dropdown_container = widgets.HBox(
    [dropdown],
    layout=widgets.Layout(
        justify_content="center",
        width="100%"
    )
)

# Toggles for header + pixel data + B magnitude + linearity check
toggle_header = widgets.ToggleButton(
    value=False,
    description="Show Header",
    layout=widgets.Layout(width="160px")
)

toggle_data = widgets.ToggleButton(
    value=False,
    description="Show Pixel Data [SLOW]",
    layout=widgets.Layout(width="180px")
)

toggle_bmag = widgets.ToggleButton(
    value=False,
    description="Show B Magnitudes",
    layout=widgets.Layout(width="180px")
)

toggle_linearity = widgets.ToggleButton(
    value=False,
    description="Show Linearity Check",
    layout=widgets.Layout(width="200px")
)

toggle_header.add_class("dark-toggle")
toggle_data.add_class("dark-toggle")
toggle_bmag.add_class("dark-toggle")
toggle_linearity.add_class("dark-toggle")

# Makes each toggle flip its own label between "Show X" / "Hide X" and adds
# a brighter active style, so it's obvious at a glance which are switched on
def make_label_handler(toggle, label):
    def handler(change):
        if change["new"]:
            toggle.description = f"Hide {label}"
            toggle.add_class("dark-toggle-active")
        else:
            toggle.description = f"Show {label}"
            toggle.remove_class("dark-toggle-active")
    return handler

toggle_header.observe(make_label_handler(toggle_header, "Header"), names="value")
toggle_data.observe(make_label_handler(toggle_data, "Pixel Data"), names="value")
toggle_bmag.observe(make_label_handler(toggle_bmag, "B Magnitudes"), names="value")
toggle_linearity.observe(make_label_handler(toggle_linearity, "Linearity Check"), names="value")

toggles_container = widgets.HBox(
    [toggle_header, toggle_data, toggle_bmag, toggle_linearity],
    layout=widgets.Layout(
        justify_content="center",
        width="100%",
        margin="10px 0px"
    )
)

# Main image output
output = widgets.Output(
    layout=widgets.Layout(
        padding="10px",
        width="100%",
        display="flex",
        justify_content="center",
        align_items="center"
    )
)

output.add_class("dark-output")

output_container = widgets.HBox(
    [output],
    layout=widgets.Layout(
        justify_content="center",
        width="100%"
    )
)

# Header output
headers_output = widgets.Output(
    layout=widgets.Layout(padding="10px", width="100%")
)

headers_output.add_class("dark-output")

headers_container = widgets.HBox(
    [headers_output],
    layout=widgets.Layout(
        justify_content="center",
        width="100%"
    )
)

# Pixel data output
data_output = widgets.Output(
    layout=widgets.Layout(padding="10px", width="100%")
)

data_output.add_class("dark-output")

data_container = widgets.HBox(
    [data_output],
    layout=widgets.Layout(
        justify_content="center",
        width="100%"
    )
)

# B magnitude output
bmag_output = widgets.Output(
    layout=widgets.Layout(padding="10px", width="100%")
)

bmag_output.add_class("dark-output")

bmag_container = widgets.HBox(
    [bmag_output],
    layout=widgets.Layout(
        justify_content="center",
        width="100%"
    )
)

# Linearity / reciprocity-failure check output
linearity_output = widgets.Output(
    layout=widgets.Layout(
        padding="10px",
        width="100%",
        display="flex",
        flex_flow="column",
        align_items="center"
    )
)

linearity_output.add_class("dark-output")

linearity_container = widgets.HBox(
    [linearity_output],
    layout=widgets.Layout(
        justify_content="center",
        width="100%"
    )
)

current_path = None
current_data = None

def render_header():
    with headers_output:
        clear_output(wait=True)

        if not toggle_header.value or current_path is None:
            return

        header = fits.getheader(current_path)

        header_df = pd.DataFrame(list(header.items()), columns=['Keyword', 'Value'])

        header_html = header_df.to_html(
            classes="header-table",
            border=0,
            index=False
        )

        display(HTML("<h3 style='color:white;text-align:center;'>FITS Header</h3>"))
        display(HTML(f'<div class="header-scroll-box">{header_html}</div>'))

def render_data():
    with data_output:
        clear_output(wait=True)

        if not toggle_data.value or current_data is None:
            return

        df = pd.DataFrame(current_data)
        table_html = df.to_html(classes="dark-table", border=0)

        display(HTML("<h3 style='color:white;text-align:center;'>Pixel Data</h3>"))
        display(HTML(f'<div class="table-scroll-box">{table_html}</div>'))

def render_bmag():
    with bmag_output:
        clear_output(wait=True)

        if not toggle_bmag.value or current_path is None:
            return

        filename = current_path.name
        entry = bmag_lookup.get(filename)

        display(HTML("<h3 style='color:white;text-align:center;'>APASS B Magnitude Summary</h3>"))

        if entry is None:
            display(HTML(f"""
            <div style="color:white; font-family: monospace; text-align:center;">
                <p><b>Plate:</b> {filename}</p>
                <p>This filename was not found in the photometry CSV at all.</p>
            </div>
            """))
            return

        bmag_value = entry["plate_apass_B_mag"]
        n_matched = entry["n_matched_stars"]

        if pd.isna(bmag_value) or n_matched == 0:
            note = "Plate is listed in the CSV, but 0 stars were matched against APASS, so no B magnitude could be computed."
        else:
            note = ""

        display(HTML(f"""
        <div style="color:white; font-family: monospace; text-align:center;">
            <p><b>Plate:</b> {filename}</p>
            <p><b>B magnitude (median matched):</b> {bmag_value}</p>
            <p><b>Matched stars:</b> {n_matched}</p>
            {f"<p style='color:#f5a623;'>{note}</p>" if note else ""}
        </div>
        """))

def render_linearity():
    with linearity_output:
        clear_output(wait=True)

        if not toggle_linearity.value or current_data is None:
            return

        if not PHOTUTILS_AVAILABLE:
            display(HTML("""
            <div style="color:#f5a623; font-family: monospace; text-align:center;">
                <p>photutils is required for this check.</p>
                <p>Run <b>pip install photutils</b> and re-run this cell.</p>
            </div>
            """))
            return

        data = current_data.astype(float)

        # Robust background stats, then detect star-like sources
        mean, median, std = sigma_clipped_stats(data, sigma=3.0)
        daofind = DAOStarFinder(fwhm=3.0, threshold=5.0 * std)
        sources = daofind(data - median)

        if sources is None or len(sources) == 0:
            display(HTML("<h3 style='color:white;text-align:center;'>Linearity / Reciprocity-Failure Check</h3>"))
            display(HTML("<p style='color:white; text-align:center;'>No sources detected above the threshold on this plate.</p>"))
            return

        # Newer photutils renamed 'xcentroid'/'ycentroid' to 'x_centroid'/
        # 'y_centroid' and warns (in unreadable black stderr text) if you
        # use the old names -- pick whichever actually exists so this works
        # across versions without ever triggering that warning.
        xcol = "x_centroid" if "x_centroid" in sources.colnames else "xcentroid"
        ycol = "y_centroid" if "y_centroid" in sources.colnames else "ycentroid"

        positions = np.transpose((sources[xcol], sources[ycol]))

        aperture_r = 5.0
        bkg_in = 8.0
        bkg_out = 12.0

        apertures = CircularAperture(positions, r=aperture_r)
        annulus_apertures = CircularAnnulus(positions, r_in=bkg_in, r_out=bkg_out)

        phot_table = aperture_photometry(data, apertures)
        bkg_stats = ApertureStats(data, annulus_apertures)

        bkg_total = bkg_stats.median * apertures.area
        total_counts = phot_table["aperture_sum"] - bkg_total
        peak_values = sources["peak"]  # already roughly bkg-subtracted, from data - median

        # Guard against non-positive values before taking logs
        valid = (total_counts > 0) & (peak_values > 0)
        total_counts_valid = np.asarray(total_counts[valid])
        peak_values_valid = np.asarray(peak_values[valid])

        display(HTML("<h3 style='color:white;text-align:center;'>Linearity / Reciprocity-Failure Check</h3>"))

        if valid.sum() < 2:
            display(HTML("<p style='color:white; text-align:center;'>Not enough valid sources to fit a linearity trend.</p>"))
        else:
            log_counts = np.log10(total_counts_valid)
            log_peak = np.log10(peak_values_valid)

            slope, intercept = np.polyfit(log_counts, log_peak, 1)
            fit_x = np.linspace(log_counts.min(), log_counts.max(), 100)
            fit_y = slope * fit_x + intercept

            fig, ax = plt.subplots(figsize=(8, 6), facecolor="#111111")
            ax.set_facecolor("#111111")

            ax.scatter(log_counts, log_peak, color="cyan", s=25, label="Detected stars")
            ax.plot(fit_x, fit_y, color="orange", linewidth=2,
                    label=f"Fit slope = {slope:.3f}")
            ax.plot(fit_x, fit_x + (log_peak.mean() - log_counts.mean()),
                    color="white", linestyle="--", linewidth=1, label="Ideal slope = 1")

            ax.set_xlabel("log10(Total aperture counts)", color="white")
            ax.set_ylabel("log10(Peak pixel value)", color="white")
            ax.set_title(f"{current_path.stem} — {len(total_counts_valid)} stars", color="white")
            ax.tick_params(colors="white")
            for spine in ax.spines.values():
                spine.set_color("white")
            legend = ax.legend(facecolor="#222222", edgecolor="white")
            for text in legend.get_texts():
                text.set_color("white")

            buf = BytesIO()
            fig.savefig(buf, format="png", facecolor=fig.get_facecolor(), bbox_inches="tight")
            plt.close(fig)
            buf.seek(0)

            display(widgets.Image(value=buf.read(), format="png"))

            note = ("Slope ≈ 1 indicates a linear plate response. A slope noticeably "
                    "below 1 (peak counts flattening relative to total counts) is "
                    "consistent with reciprocity failure / saturation, typically at "
                    "the bright end.")
            display(HTML(f"<p style='color:white; text-align:center; max-width:700px; margin:auto;'>{note}</p>"))

        # Per-star counts table, as requested ("counts per star for all stars")
        star_table = pd.DataFrame({
            "star_id": sources["id"],
            "x": np.round(np.asarray(sources[xcol]), 2),
            "y": np.round(np.asarray(sources[ycol]), 2),
            "peak": np.round(np.asarray(peak_values), 2),
            "total_counts": np.round(np.asarray(total_counts), 2)
        })

        star_table_html = star_table.to_html(classes="dark-table", border=0, index=False)

        display(HTML("<h4 style='color:white;text-align:center;'>Per-Star Counts</h4>"))
        display(HTML(f'<div class="table-scroll-box">{star_table_html}</div>'))

def show_plate(change):
    global current_path, current_data

    current_path = change["new"]
    current_data = fits.getdata(current_path)

    with output:
        clear_output(wait=True)

        fig = plt.figure(figsize=(12, 10), facecolor="#111111")

        gs = gridspec.GridSpec(1, 3, width_ratios=[0.05, 1.0, 0.05], wspace=0.03)

        ax_spacer = fig.add_subplot(gs[0])
        ax = fig.add_subplot(gs[1])
        cax = fig.add_subplot(gs[2])

        ax_spacer.axis("off")

        fig.patch.set_facecolor("#111111")
        ax.set_facecolor("#111111")

        im = ax.imshow(current_data, origin="lower", cmap="gray")

        cbar = fig.colorbar(im, cax=cax)
        cbar.set_label("Plate Intensity", color="white", fontsize=14)
        cbar.ax.tick_params(colors="white")

        ax.set_title(current_path.stem, fontsize=28, color="white", pad=16)
        ax.set_xlabel("X Pixel", color="white")
        ax.set_ylabel("Y Pixel", color="white")
        ax.tick_params(colors="white")

        buf = BytesIO()
        fig.savefig(buf, format="png", facecolor=fig.get_facecolor(), bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)

        display(widgets.Image(value=buf.read(), format="png"))

    render_header()
    render_data()
    render_bmag()
    render_linearity()

dropdown.observe(show_plate, names="value")
toggle_header.observe(lambda change: render_header(), names="value")
toggle_data.observe(lambda change: render_data(), names="value")
toggle_bmag.observe(lambda change: render_bmag(), names="value")
toggle_linearity.observe(lambda change: render_linearity(), names="value")

display(
    dropdown_container,
    toggles_container,
    output_container,
    headers_container,
    data_container,
    bmag_container,
    linearity_container
)

# Show first plate automatically
if cutouts:
    show_plate({"new": cutouts[0]})

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\dapur\\Downloads\\Other\\Research\\rcb_Star_Dust_Survey\\test_CNN\\data\\raw_daschlab\\R_CrB\\R_CrB_B_magnitudes.csv'

In [ ]:
# TEST

from pathlib import Path
from io import BytesIO
from astropy.io import fits
from astropy.time import Time
from astropy.stats import sigma_clipped_stats
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# photutils is needed for the linearity/reciprocity-failure check (star
# detection + aperture photometry). Guarded import so the rest of the
# notebook still works if it's not installed -- pip install photutils
try:
    from photutils.detection import DAOStarFinder
    from photutils.aperture import CircularAperture, CircularAnnulus, ApertureStats, aperture_photometry
    PHOTUTILS_AVAILABLE = True
except ImportError:
    PHOTUTILS_AVAILABLE = False
    print("photutils not found -- run 'pip install photutils' to enable the Linearity Check toggle.")

# Dark styling for the dropdown + toggles + table + kill any default borders/backgrounds/outlines
display(HTML("""
<style>

/* =========================
   FIXED DROPDOWN STYLING
   ========================= */

/* main dropdown box */
.dark-dropdown select {
    background-color: #222 !important;   /* darker so text is visible */
    color: white !important;
    border: 1px solid #555 !important;
}

/* dropdown arrow area */
.dark-dropdown {
    color: white !important;
}

/* options list (this is what was invisible before) */
.dark-dropdown option {
    background-color: #222 !important;
    color: white !important;
}

/* hover behavior inside dropdown list */
.dark-dropdown option:hover {
    background-color: #444 !important;
    color: white !important;
}

.dark-toggle {
    color: white !important;
    background-color: #333 !important;
    border: 1px solid #777 !important;
    font-weight: bold !important;
    font-size: 13px !important;
    padding: 6px 10px !important;
}

.dark-toggle:hover {
    background-color: #4a4a4a !important;
}

.dark-toggle-active {
    background-color: #2e7d32 !important;
    border: 1px solid #66bb6a !important;
}

.dark-toggle-active:hover {
    background-color: #388e3c !important;
}

.dark-output,
.dark-output .output,
.dark-output .widget-output,
.jp-OutputArea-output,
.jp-OutputArea-child,
.cell-output-ipywidget-background,
.output,
.output_wrapper,
.widgetarea,
.jp-RenderedHTMLCommon,
.jupyter-widgets {
    background-color: #111 !important;
    border: none !important;
    box-shadow: none !important;
    outline: none !important;
}

body, .jp-Notebook, .jp-WindowedPanel-outer, .jp-Cell-outputArea {
    background-color: #111 !important;
}

.dark-table {
    border-collapse: collapse;
    color: #ddd;
    background-color: #111;
    font-family: monospace;
    font-size: 11px;
}

.dark-table th,
.dark-table td {
    border: 1px solid #333;
    padding: 2px 6px;
    text-align: right;
}

.dark-table th {
    background-color: #222;
    color: white;
}

.table-scroll-box {
    max-height: 500px;
    max-width: 100%;
    overflow: auto;
    background-color: #111;
}

.header-table {
    border-collapse: collapse;
    color: #ddd;
    background-color: #111;
    font-family: monospace;
    font-size: 12px;
    width: 100%;
}

.header-table th,
.header-table td {
    border: 1px solid #333;
    padding: 4px 10px;
    text-align: left;
}

.header-table th {
    background-color: #222;
    color: white;
}

.header-scroll-box {
    max-height: 300px;
    overflow: auto;
    background-color: #111;
    margin-bottom: 20px;
}

</style>
"""))

cutout_dir = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB\cutouts"
)

csv_path = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB\R_CrB_B_magnitudes.csv"
)

photometry_df = pd.read_csv(csv_path)

cutouts = sorted(cutout_dir.glob("*.fits"))[:50]
print(f"Found {len(cutouts)} cutouts")

# =========================
# dropdown options
# =========================
photometry_df["fits_file"] = photometry_df["fits_file"].astype(str).str.strip()
bmag_lookup = photometry_df.set_index("fits_file")[["plate_apass_B_mag", "n_matched_stars"]].to_dict("index")

dropdown_options = []

for f in cutouts:
    filename = str(f.name).strip()
    entry = bmag_lookup.get(filename)

    if entry is None:
        bmag_label = "✖ Not in CSV"
        linearity_label = "✖ No data"
    elif pd.isna(entry["plate_apass_B_mag"]) or entry["n_matched_stars"] == 0:
        bmag_label = "△ In CSV, no match"
        linearity_label = "✖ No stars"
    else:
        bmag_label = "✔ Bmag"
        n_matched = entry["n_matched_stars"]
        linearity_label = "✔ Linearity OK" if n_matched >= 3 else "△ Few stars"

    dropdown_options.append(
        (f"{filename}  |  {bmag_label}  |  {linearity_label}", f)
    )

dropdown = widgets.Dropdown(
    options=dropdown_options,
    description="",
    layout=widgets.Layout(width="1200px")
)

dropdown.add_class("dark-dropdown")

dropdown_container = widgets.HBox(
    [dropdown],
    layout=widgets.Layout(justify_content="center", width="100%")
)

# =========================
# rest of your UI (UNCHANGED)
# =========================
toggle_header = widgets.ToggleButton(value=False, description="Show Header", layout=widgets.Layout(width="160px"))
toggle_data = widgets.ToggleButton(value=False, description="Show Pixel Data [SLOW]", layout=widgets.Layout(width="180px"))
toggle_bmag = widgets.ToggleButton(value=False, description="Show B Magnitudes", layout=widgets.Layout(width="180px"))
toggle_linearity = widgets.ToggleButton(value=False, description="Show Linearity Check", layout=widgets.Layout(width="200px"))

toggle_header.add_class("dark-toggle")
toggle_data.add_class("dark-toggle")
toggle_bmag.add_class("dark-toggle")
toggle_linearity.add_class("dark-toggle")

def make_label_handler(toggle, label):
    def handler(change):
        if change["new"]:
            toggle.description = f"Hide {label}"
            toggle.add_class("dark-toggle-active")
        else:
            toggle.description = f"Show {label}"
            toggle.remove_class("dark-toggle-active")
    return handler

toggle_header.observe(make_label_handler(toggle_header, "Header"), names="value")
toggle_data.observe(make_label_handler(toggle_data, "Pixel Data"), names="value")
toggle_bmag.observe(make_label_handler(toggle_bmag, "B Magnitudes"), names="value")
toggle_linearity.observe(make_label_handler(toggle_linearity, "Linearity Check"), names="value")

toggles_container = widgets.HBox(
    [toggle_header, toggle_data, toggle_bmag, toggle_linearity],
    layout=widgets.Layout(justify_content="center", width="100%", margin="10px 0px")
)

# Outputs (unchanged)
output = widgets.Output(layout=widgets.Layout(padding="10px", width="100%", justify_content="center"))
output.add_class("dark-output")

output_container = widgets.HBox([output], layout=widgets.Layout(justify_content="center", width="100%"))

headers_output = widgets.Output(layout=widgets.Layout(padding="10px", width="100%"))
headers_output.add_class("dark-output")
headers_container = widgets.HBox([headers_output], layout=widgets.Layout(justify_content="center", width="100%"))

data_output = widgets.Output(layout=widgets.Layout(padding="10px", width="100%"))
data_output.add_class("dark-output")
data_container = widgets.HBox([data_output], layout=widgets.Layout(justify_content="center", width="100%"))

bmag_output = widgets.Output(layout=widgets.Layout(padding="10px", width="100%"))
bmag_output.add_class("dark-output")
bmag_container = widgets.HBox([bmag_output], layout=widgets.Layout(justify_content="center", width="100%"))

linearity_output = widgets.Output(layout=widgets.Layout(padding="10px", width="100%", display="flex", flex_flow="column", align_items="center"))
linearity_output.add_class("dark-output")

linearity_container = widgets.HBox([linearity_output], layout=widgets.Layout(justify_content="center", width="100%"))

current_path = None
current_data = None

# =========================
# IMPORTANT: everything below unchanged
# (render functions + show_plate)
# =========================
# (keep your existing render_header / render_data / render_bmag / render_linearity / show_plate exactly as-is)

dropdown.observe(show_plate, names="value")

display(
    dropdown_container,
    toggles_container,
    output_container,
    headers_container,
    data_container,
    bmag_container,
    linearity_container
)

if cutouts:
    show_plate({"new": cutouts[0]})

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\dapur\\Downloads\\Other\\Research\\rcb_Star_Dust_Survey\\test_CNN\\data\\raw_daschlab\\R_CrB\\R_CrB_B_magnitudes.csv'

In [4]:
from pathlib import Path
import pandas as pd
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.vizier import Vizier

# Check if cutouts exist
cutout_dir = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw_daschlab\R_CrB\cutouts")
cutouts = sorted(cutout_dir.glob("*.fits"))

print(f"=== DIAGNOSTIC CHECK ===")
print(f"Number of cutout FITS files: {len(cutouts)}")

if len(cutouts) > 0:
    # Test one plate
    test_file = cutouts[0]
    print(f"\nTesting: {test_file.name}")
    
    # Read the image
    hdu = fits.open(test_file)[0]
    image = hdu.data
    wcs = WCS(hdu.header)
    
    # Check if WCS is valid
    print(f"WCS: CRVAL1={hdu.header.get('CRVAL1', 'MISSING')}, CRVAL2={hdu.header.get('CRVAL2', 'MISSING')}")
    
    # Try star detection
    mean, median, std = sigma_clipped_stats(image)
    print(f"Image stats - Mean: {mean:.2f}, Median: {median:.2f}, Std: {std:.2f}")
    
    finder = DAOStarFinder(threshold=5.0 * std, fwhm=3.0)
    sources = finder(image - median)
    
    if sources is not None:
        print(f"Found {len(sources)} sources")
        if len(sources) > 0:
            print(f"First source: x={sources['x_centroid'][0]:.2f}, y={sources['y_centroid'][0]:.2f}")
    else:
        print("Found NO sources!")
        
    # Test APASS connection
    print("\nTesting APASS catalog query...")
    center = SkyCoord(hdu.header["CRVAL1"], hdu.header["CRVAL2"], unit="deg")
    
    try:
        Vizier.ROW_LIMIT = -1
        result = Vizier.query_region(center, radius=10 * u.arcmin, catalog="II/336/apass9")
        
        if len(result) > 0:
            apass = result[0]
            print(f"✅ APASS query SUCCESS: Found {len(apass)} stars")
            print(f"Sample Bmag: {apass['Bmag'][0] if len(apass) > 0 else 'N/A'}")
        else:
            print("❌ APASS query returned NO data")
    except Exception as e:
        print(f"❌ APASS query FAILED: {e}")
else:
    print("\n❌ No cutouts found! Cell 1 didn't download any images.")
    print("This could be due to:")
    print("  - Internet connection issues")
    print("  - daschlab session not working properly")
    print("  - Target 'R CrB' not found in the database")
    print("  - Permission issues with the directory")

=== DIAGNOSTIC CHECK ===
Number of cutout FITS files: 57

Testing: 00012_i00936m1s0.fits
WCS: CRVAL1=237.14339784, CRVAL2=28.15674885
Image stats - Mean: 3251.83, Median: 3850.00, Std: 1592.40
Found NO sources!

Testing APASS catalog query...
✅ APASS query SUCCESS: Found 26 stars
Sample Bmag: 15.539999961853027
